<a href="https://colab.research.google.com/github/vishal9198/genAi-Labs/blob/main/routing_in_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-openai langchain-community langchain-core pydantic

In [ ]:
import os
from google.colab import userdata

# -------------------------------------------------------------
# 0. API KEY CONFIGURATION
# -------------------------------------------------------------
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# =============================================================
# PART 1: LOGICAL ROUTING (LLM Function Calling)
# =============================================================
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

# Step 1: Define the target data sources via Pydantic
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question, choose which datasource would be most relevant for answering their question",
    )

# Step 2: Initialize LLM with structured output constraint
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

# Step 3: Define routing instructions
system_prompt = """You are an expert at routing a user question to the appropriate data source.
Based on the programming language the question is referring to, route it to the relevant data source."""

prompt_logical = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

# Step 4: Build router chain
router = prompt_logical | structured_llm

# Step 5: Define branching logic (downstream destination)
def choose_route(result: RouteQuery):
    if "python_docs" in result.datasource.lower():
        # Insert specific Python retriever/chain here
        return "Executing >> CHAIN FOR PYTHON_DOCS"
    elif "js_docs" in result.datasource.lower():
        # Insert specific JavaScript retriever/chain here
        return "Executing >> CHAIN FOR JS_DOCS"
    else:
        # Insert specific Golang retriever/chain here
        return "Executing >> CHAIN FOR GOLANG_DOCS"

full_logical_chain = router | RunnableLambda(choose_route)

# Test Logical Routing
logical_question = """why doesn't the following code work:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")"""

print("--- LOGICAL ROUTING OUTPUT ---")
print("Target Chosen:", router.invoke({"question": logical_question}))
print(full_logical_chain.invoke({"question": logical_question}))
print("\n" + "=" * 60 + "\n")


# =============================================================
# PART 2: SEMANTIC ROUTING (Embedding Cosine Similarity)
# =============================================================
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings

# Step 1: Define domain prompt templates
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

prompt_templates = [physics_template, math_template]

# Step 2: Embed the prompt definitions
embeddings = OpenAIEmbeddings()
prompt_embeddings = embeddings.embed_documents(prompt_templates)

# Step 3: Define routing function based on vector similarity
def prompt_router(input_dict):
    # Embed the incoming user query
    query_embedding = embeddings.embed_query(input_dict["query"])

    # Compute cosine similarity against pre-computed prompt embeddings
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]

    # Select the prompt with the highest similarity score
    most_similar = prompt_templates[similarity.argmax()]

    # Log which route was taken
    if most_similar == math_template:
        print("[Router Decision]: Selected MATH Prompt via Vector Similarity")
    else:
        print("[Router Decision]: Selected PHYSICS Prompt via Vector Similarity")

    return PromptTemplate.from_template(most_similar)

# Step 4: Construct LCEL Chain
semantic_chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatOpenAI(temperature=0)
    | StrOutputParser()
)

# Test Semantic Routing
print("--- SEMANTIC ROUTING OUTPUT ---")
semantic_query = "What's a black hole?"
print("Query:", semantic_query)
response = semantic_chain.invoke(semantic_query)
print("\nFinal Model Response:")
print(response)